# Simple kappa -> shear forward-model demo

This notebook demonstrates the starter workflow:
1. load kappa maps,
2. convert to clean shear,
3. add COSMOS mask and noise,
4. plot the results.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

repo_root = Path.cwd()
if not (repo_root / "data").exists() and (repo_root.parent / "data").exists():
    repo_root = repo_root.parent

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from operators import kappa_to_shear
from noise_mask import load_noise_mask, apply_mask_and_noise

In [ ]:
kappa_file = repo_root / "data" / "kappa_subset_lp001_runs001-010_wlmmuq_384.npz"
if not kappa_file.exists():
    kappa_file = repo_root / "data" / "kappa_subset_lp001_runs001-010_wlmmuq_256.npz"

payload = np.load(kappa_file)
kappa = np.asarray(payload["kappa"], dtype=np.float32)

print(f"Loaded {kappa_file}")
print("kappa shape:", kappa.shape)

In [ ]:
map_size = int(kappa.shape[-1])
noise_mask_file = repo_root / "data" / f"cosmos_noise_mask_{map_size}.npz"
std_noise, mask, extent = load_noise_mask(str(noise_mask_file))

print(f"Loaded {noise_mask_file}")
print("std_noise shape:", std_noise.shape)
print("mask valid fraction:", float(mask.mean()))

In [ ]:
gamma_clean = kappa_to_shear(kappa, return_complex=True)
gamma_noisy, noise = apply_mask_and_noise(
    gamma_clean,
    std_noise=std_noise,
    mask=mask,
    seed=42,
    inpainting=False,
    return_noise=True,
)

print("gamma_clean shape:", gamma_clean.shape)
print("gamma_noisy shape:", gamma_noisy.shape)

In [ ]:
def symmetric_limits(arr, q=99):
    vmax = float(np.nanpercentile(np.abs(arr), q))
    if not np.isfinite(vmax) or vmax <= 0:
        vmax = 1e-6
    return -vmax, vmax

idx = 0
g1_clean = np.real(gamma_clean[idx])
g2_clean = np.imag(gamma_clean[idx])
g1_noisy = np.real(gamma_noisy[idx])
g2_noisy = np.imag(gamma_noisy[idx])
n1 = np.real(noise[idx])

kmin, kmax = symmetric_limits(kappa[idx])
gmin, gmax = symmetric_limits(np.stack([g1_clean, g2_clean, g1_noisy, g2_noisy], axis=0))
nmin, nmax = symmetric_limits(n1)
sn_max = float(np.nanpercentile(std_noise, 99.5))

fig, axes = plt.subplots(2, 4, figsize=(16, 8), constrained_layout=True)
panels = [
    (kappa[idx], "kappa", kmin, kmax),
    (g1_clean, "gamma1 clean", gmin, gmax),
    (g2_clean, "gamma2 clean", gmin, gmax),
    (std_noise, "std_noise", 0.0, sn_max),
    (g1_noisy, "gamma1 noisy", gmin, gmax),
    (g2_noisy, "gamma2 noisy", gmin, gmax),
    (n1, "noise gamma1", nmin, nmax),
    (mask.astype(float), "mask", 0.0, 1.0),
]

for ax, (arr, title, vmin, vmax) in zip(axes.flat, panels):
    im = ax.imshow(arr, cmap="viridis", vmin=vmin, vmax=vmax)
    ax.set_title(title)
    ax.axis("off")
    fig.colorbar(im, ax=ax, shrink=0.8)

fig.suptitle(f"Forward model example (map {idx})", fontsize=13)
plt.show()

In [ ]:
n_show = min(10, kappa.shape[0])
ncols = 5
nrows = int(np.ceil(n_show / ncols))

vmin = float(np.nanpercentile(kappa[:n_show], 1.0))
vmax = float(np.nanpercentile(kappa[:n_show], 99.0))

fig, axes = plt.subplots(nrows, ncols, figsize=(3.5 * ncols, 3.0 * nrows), constrained_layout=True)
axes = np.atleast_1d(axes).ravel()

for i, ax in enumerate(axes):
    if i >= n_show:
        ax.axis("off")
        continue
    im = ax.imshow(kappa[i], cmap="viridis", vmin=vmin, vmax=vmax)
    ax.set_title(f"kappa[{i}]")
    ax.axis("off")

fig.colorbar(im, ax=axes.tolist(), shrink=0.8)
plt.show()